# 🎙️ VoiceBatch Studio v0.0 - [Google Drive Local Mode]
यह कोड सीधे आपके ड्राइव में मौजूद मॉडल का इस्तेमाल करेगा ताकि डाउनलोडिंग में समय बर्बाद न हो।

In [ ]:
# @title 🛠️ Step 1: ड्राइव कनेक्ट और लाइब्रेरी सेटअप
import os
from google.colab import drive
from IPython.display import display, Javascript

# Anti-Sleep Script
display(Javascript('function ClickConnect(){document.querySelector("colab-connect-button").click()}setInterval(ClickConnect,60000)'))

print("⏳ लाइब्रेरीज़ सेटअप हो रही हैं...")
!pip uninstall -q coqpit coqui-tts -y
!pip install -q coqpit-config coqui-tts gradio librosa soundfile

print("🔓 कृपया ड्राइव परमिशन दें ताकि मॉडल लोड हो सके...")
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

os.makedirs("outputs", exist_ok=True)
print("✅ ड्राइव कनेक्ट हो गई!")

In [ ]:
# @title 🚀 Step 2: ड्राइव मॉडल से स्टूडियो लॉन्च करें
import gradio as gr
import torch, librosa, re, numpy as np, soundfile as sf
from TTS.api import TTS

device = 'cuda' if torch.cuda.is_available() else 'cpu'
# आपके ड्राइव का वही पक्का रास्ता
model_path = "/content/drive/MyDrive/VoiceBatchModels/"

if os.path.exists(model_path + "model.pth"):
    print("✅ ड्राइव में मॉडल मिल गया! लोड हो रहा है...")
    tts = TTS(model_path=model_path, config_path=model_path + "config.json").to(device)
    print("🚀 इंजन तैयार है!")
else:
    print("❌ एरर: आपके ड्राइव में 'VoiceBatchModels' फोल्डर या फाइलें नहीं मिलीं।")

def voice_gen(text, audio_sample):
    # अनलिमिटेड स्क्रिप्ट को टुकड़ों में प्रोसेस करना
    parts = re.split(r'(?<=[।?!])\s+', text)
    combined = []
    for p in parts:
        if len(p.strip()) < 2: continue
        tts.tts_to_file(text=p, speaker_wav=audio_sample, language='hi', file_path='temp.wav')
        y, _ = librosa.load('temp.wav', sr=24000)
        combined.extend(y)
    
    output_path = "outputs/VoiceBatch_Result.wav"
    sf.write(output_path, np.array(combined), 24000)
    return output_path

gr.Interface(fn=voice_gen, 
             inputs=[gr.Textbox(label="पूरी कहानी यहाँ डालें (कोई लिमिट नहीं)", lines=10), 
                     gr.Audio(label="अपना वॉयस सैंपल यहाँ अपलोड करें", type='filepath')],
             outputs=gr.Audio(label="ऑडियो डाउनलोड करें")).launch(share=True)